# Multitienda del Sur SpA - Pipeline Entrega 1

---


**IIC 3752 - Grupo 3** - Pregunta 1: *¿La cadena está reteniendo a sus clientes o los está reponiendo con compradores de una sola vez?*

El notebook es la **cadena completa** de la Entrega 1 y se ejecuta de principio a fin con *Entorno de ejecución => Ejecutar todas*.
Comienza con la **Etapa 1: extracción** (capa bronce). Las etapas siguientes (limpieza, modelo dimensional, métrica y
visualizaciones) están a continuación en este mismo notebook.

**Etapa 1.** Se leen las cuatro fuentes del caso con **Apache Spark (PySpark)** y cada tabla se guarda en **Parquet** en una
zona raw particionada por fecha de carga: `raw/{tabla}/{id_fecha}/`.

| # | Fuente | Formato | Tablas raw |
|---|---|---|---|
| 1 | `origen_pos.sqlite` | Base SQLite del POS | `pos_boletas`, `pos_boleta_detalle`, `pos_clientes`, `pos_tiendas`, `pos_comunas`, `pos_regiones`, `pos_medios_pago` |
| 2 | `origen_maestro_productos.csv` | CSV (`;`, cp1252) | `productos` |
| 3 | `origen_promociones.json` | JSON anidado | `promo_campanas`, `promo_campana_sku` |
| 4 | `mindicador.cl/api/ipc/2025` | API REST | `ipc_mensual` |

**Criterio de la capa raw.** El dato se guarda *tal como llega*: no se limpia, no se deduplica y no se cambian tipos. Las
columnas del POS se guardan como texto para no perder información en una conversión silenciosa (por ejemplo, `"3.176"` leído
como número 3,176 o un `cliente_id` vacío convertido en `364.0`). Limpieza ocurre en la etapa siguiente, leyendo
desde aquí. A cada tabla se le agregan dos columnas de linaje: `_fuente` y `_id_fecha_carga`.

## 0. Setup
En Colab se instala PySpark (la misma versión que se uso en la ayudantía; Colab ya trae Java). Fuera de Colab, se debe instalar antes
`requirements.txt` y tener Java 17 (ver README del repositorio). La primera ejecución toma 1 a 2 minutos.

In [ ]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)
print("Ejecutando en Colab" if EN_COLAB else "Ejecutando en entorno local")

In [ ]:
import os, json, shutil, sqlite3, time
from datetime import date

import requests
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = (SparkSession.builder
         .appName("multitienda-e1")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "America/Santiago")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "listo")

### Archivos de entrada
Los tres archivos del caso van en la carpeta `data/`. Si no están, en Colab se abre el diálogo para subirlos (mientras que la API del
IPC se descarga sola).

In [ ]:
DATA_DIR = "data"
FUENTES = {
    "pos":         "origen_pos.sqlite",
    "productos":   "origen_maestro_productos.csv",
    "promociones": "origen_promociones.json",
}
os.makedirs(DATA_DIR, exist_ok=True)

def ruta_dato(clave):
    return os.path.join(DATA_DIR, FUENTES[clave])

def mover_a_data():
    # Mover archivos a data/ en caso de estar en la carpeta actual
    for archivo in FUENTES.values():
        if os.path.exists(archivo) and not os.path.exists(os.path.join(DATA_DIR, archivo)):
            shutil.move(archivo, os.path.join(DATA_DIR, archivo))

def faltantes():
    return [a for a in FUENTES.values() if not os.path.exists(os.path.join(DATA_DIR, a))]

mover_a_data()
if faltantes() and EN_COLAB:
    from google.colab import files
    print("Sube estos archivos:", faltantes())
    for nombre in files.upload():
        shutil.move(nombre, nombre.split("/")[-1])
    mover_a_data()

assert not faltantes(), f"Faltan archivos en {DATA_DIR}/: {faltantes()}"
for clave in FUENTES:
    print(f"OK  {ruta_dato(clave):<40} {os.path.getsize(ruta_dato(clave)):>9,} bytes")

### Zona raw y bitácora de carga
`guardar_raw` escribe cada tabla en `raw/{tabla}/{ID_FECHA}/`, la relee para confirmar cuántas filas quedaron y lo anota
en una bitácora de carga, siendo esta el punto de partida de la **cobertura de datos** pedida en el informe. Todo lo
que después se excluya se mide contra estas filas.

In [ ]:
ID_FECHA = date.today().strftime("%Y%m%d")
RAW = "raw"
bitacora_carga = []

def guardar_raw(df, tabla, fuente, filas_origen=None):
    df = (df.withColumn("_fuente", F.lit(fuente))
            .withColumn("_id_fecha_carga", F.lit(ID_FECHA)))
    ruta = f"{RAW}/{tabla}/{ID_FECHA}"
    df.write.mode("overwrite").parquet(ruta)
    filas = spark.read.parquet(ruta).count()
    if filas_origen is not None:
        assert filas == filas_origen, f"{tabla}: {filas} filas en raw vs {filas_origen} en origen"
    bitacora_carga.append({"fuente": fuente, "tabla": tabla, "filas": filas,
                           "columnas": len(df.columns) - 2, "ruta": ruta})
    print(f"{tabla:<20} {filas:>7,} filas  ->  {ruta}")
    return filas

print("id_fecha de carga:", ID_FECHA)

## 1. Fuente SQL: base del POS (SQLite)
Dado que Spark no lee SQLite directamente, cada tabla se lee con la librería
`sqlite3` y se pasa a Spark. Se recorren **todas** las tablas de la base, de modo que si el origen agrega una tabla nueva,
también entra. Cada valor se convierte a texto tal como está almacenado y los nulos se mantienen como nulos.

In [ ]:
con = sqlite3.connect(ruta_dato("pos"))
tablas_pos = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print("Tablas en el POS:", tablas_pos, "\n")

for t in tablas_pos:
    cur = con.execute(f'SELECT * FROM "{t}"')
    columnas = [d[0] for d in cur.description]
    filas = [tuple(None if v is None else str(v) for v in fila) for fila in cur.fetchall()]
    esquema = StructType([StructField(c, StringType(), True) for c in columnas])
    df = spark.createDataFrame(filas, esquema)
    guardar_raw(df, f"pos_{t}", FUENTES["pos"], filas_origen=len(filas))

con.close()

In [ ]:
# Revisión de tabla central: se notan fechas en varios formatos y boletas sin cliente (anónimas)
spark.read.parquet(f"{RAW}/pos_boletas/{ID_FECHA}").show(8, truncate=False)

## 2. Fuente CSV: maestro de productos
Si bien el archivo viene separado por `;` y codificado en Windows-1252, Spark no acepta el nombre `windows-1252`, pero
`iso-8859-1` decodifica igual los acentos de este archivo. No se infiere el esquema: todo entra como texto, porque los
precios vienen en formatos mezclados (`1.223,00`, `$1.213`, `1.921`) y una inferencia automática podría dañarlos.

In [ ]:
productos = (spark.read
             .option("header", True)
             .option("sep", ";")
             .option("encoding", "iso-8859-1")
             .option("inferSchema", False)
             .csv(ruta_dato("productos")))

# Punto de control: mismas filas que una lectura independiente con pandas
filas_csv = len(pd.read_csv(ruta_dato("productos"), sep=";", encoding="cp1252", dtype=str))
guardar_raw(productos, "productos", FUENTES["productos"], filas_origen=filas_csv)
productos.show(6, truncate=False)

## 3. Fuente JSON: campañas de promoción
El JSON trae una lista de campañas y, dentro de cada una, su lista de SKUs y canales. Se guardan dos tablas:
`promo_campanas` (una fila por campaña, con los canales como lista) y `promo_campana_sku` (una fila por campaña y SKU,
aplanada con `explode`), que es la que después se cruza con las ventas.

Observacion: como `descuento_pct` viene como `"25%"` en una campaña y como `50` en la otra. Spark lo lee como texto y así se guarda.

In [ ]:
promo = spark.read.option("multiline", True).json(ruta_dato("promociones"))
promo.printSchema()

campanas = (promo
    .select("cadena", F.explode("campanas").alias("c"))
    .select("cadena",
            F.col("c.campana_id").alias("campana_id"),
            F.col("c.nombre").alias("nombre"),
            F.col("c.tipo").alias("tipo"),
            F.col("c.fecha_ini").alias("fecha_ini"),
            F.col("c.fecha_fin").alias("fecha_fin"),
            F.col("c.descuento_pct").cast("string").alias("descuento_pct"),
            F.col("c.canales").alias("canales"),
            F.size("c.skus").alias("n_skus")))

campana_sku = (promo
    .select(F.explode("campanas").alias("c"))
    .select(F.col("c.campana_id").alias("campana_id"), F.explode("c.skus").alias("sku")))

with open(ruta_dato("promociones"), encoding="utf-8") as f:
    _p = json.load(f)
guardar_raw(campanas, "promo_campanas", FUENTES["promociones"], filas_origen=len(_p["campanas"]))
guardar_raw(campana_sku, "promo_campana_sku", FUENTES["promociones"],
            filas_origen=sum(len(c["skus"]) for c in _p["campanas"]))
campanas.show(truncate=False)

## 4. Fuente API: IPC mensual (mindicador.cl)
Se consulta la API pública en vivo. Si responde, además de cargarla se deja una copia en `data/ipc_2025_respaldo.json`
(archivo usado en repositorio). Se considero que si la API no responde después de tres intentos, se usa esa copia y se registra en la
bitácora, para que la corrida siga siendo reproducible.

**Acerca de los datos de la API:** `valor` es la **variación porcentual mensual** del IPC (`unidad_medida =
"Porcentaje"`), no el nivel del índice. Para deflactar hay que encadenar las variaciones. Las fechas vienen en UTC
(`2025-09-01T04:00:00Z` = 1 de septiembre en Chile).

In [ ]:
URL_IPC = "https://mindicador.cl/api/ipc/2025"
RESPALDO_IPC = os.path.join(DATA_DIR, "ipc_2025_respaldo.json")

payload, origen_ipc = None, None
for intento in range(1, 4):
    try:
        resp = requests.get(URL_IPC, timeout=20)
        resp.raise_for_status()
        payload = resp.json()
        assert payload.get("serie"), "respuesta sin serie"
        origen_ipc = "api"
        break
    except Exception as e:
        print(f"Intento {intento} falló: {e}")
        time.sleep(3)

if origen_ipc == "api":
    print("API OK:", resp.status_code)
    with open(RESPALDO_IPC, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=1)
elif os.path.exists(RESPALDO_IPC):
    with open(RESPALDO_IPC, encoding="utf-8") as f:
        payload = json.load(f)
    origen_ipc = "respaldo"
    print(f"ATENCIÓN: la API no respondió; se usa la copia {RESPALDO_IPC}")
else:
    raise RuntimeError("La API del IPC no respondió y no existe data/ipc_2025_respaldo.json")

filas_ipc = [(s["fecha"], float(s["valor"]), payload.get("unidad_medida"), payload.get("codigo"))
             for s in payload["serie"]]
esquema_ipc = StructType([StructField("fecha", StringType()), StructField("valor", DoubleType()),
                          StructField("unidad_medida", StringType()), StructField("codigo", StringType())])
ipc = spark.createDataFrame(filas_ipc, esquema_ipc)
guardar_raw(ipc, "ipc_mensual", f"mindicador.cl ({origen_ipc})", filas_origen=len(filas_ipc))
ipc.orderBy("fecha").show(12, truncate=False)

## 5. Revisión de la zona raw
Bitácora de la carga y estructura de carpetas. La bitácora también se guarda en `raw/_bitacora_carga/`.

In [ ]:
bitacora = pd.DataFrame(bitacora_carga)
os.makedirs(f"{RAW}/_bitacora_carga", exist_ok=True)
bitacora.to_csv(f"{RAW}/_bitacora_carga/{ID_FECHA}.csv", index=False)
display(bitacora)

print()
for tabla in sorted(os.listdir(RAW)):
    print(f"raw/{tabla}/ -> {sorted(os.listdir(os.path.join(RAW, tabla)))}")

## 6. Calidad del dato en origen (medida, todavía sin corregir)
Tal como pide el enunciado se **meide y declara** la calidad, no se arregla en origen. Estos conteos salen directamente de la zona
raw y alimentan la sección de cobertura y la bitácora de supuestos del informe. La corrección se hace en la Etapa 2.

In [ ]:
def leer(tabla):
    return spark.read.parquet(f"{RAW}/{tabla}/{ID_FECHA}").drop("_fuente", "_id_fecha_carga")

def patron(col):
    return F.regexp_replace(F.regexp_replace(F.trim(col), "[0-9]", "9"), "[A-Za-zÁÉÍÓÚáéíóúñÑ]+", "a")

bol, det, cli = leer("pos_boletas"), leer("pos_boleta_detalle"), leer("pos_clientes")
prod, csku = leer("productos"), leer("promo_campana_sku")

n_bol = bol.count()
calidad = {
    "boletas: filas": n_bol,
    "boletas: duplicadas exactas": n_bol - bol.dropDuplicates().count(),
    "boletas: boleta_id repetido": n_bol - bol.select("boleta_id").distinct().count(),
    "boletas: sin cliente (anónimas)": bol.filter(F.col("cliente_id").isNull()).count(),
    "boletas: fecha con año 2026": bol.filter(F.trim("fecha").rlike(r"^2026-|[/-]2026$|^\d{1,2}-\d{1,2}-26$")).count(),
    "detalle: filas": det.count(),
    "detalle: (boleta_id, linea) repetido": det.count() - det.select("boleta_id", "linea").distinct().count(),
    "detalle: monto_linea negativo": det.filter(F.col("monto_linea").startswith("-")).count(),
    "clientes: filas": cli.count(),
    "productos: filas": prod.count(),
    "productos: SKU repetido": prod.count() - prod.select("sku").distinct().count(),
    "productos: filas sin formato SKU-": prod.filter(~F.col("sku").rlike(r"^SKU-\d+$")).count(),
    "promo: SKUs que no están en el maestro": csku.join(prod.select("sku").distinct(), "sku", "left_anti").count(),
}
display(pd.Series(calidad, name="conteo").to_frame())

In [ ]:
print("Formatos de fecha en boletas:")
bol.groupBy(patron(F.col("fecha")).alias("formato")).count().orderBy(F.desc("count")).show(truncate=False)

print("Formatos de descuento en el detalle:")
det.groupBy("descuento").count().orderBy(F.desc("count")).show(truncate=False)

print("Variantes de categoría en el maestro de productos:")
prod.groupBy("categoria").count().orderBy("categoria").show(50, truncate=False)

print("Valores de 'activo' en el maestro de productos:")
prod.groupBy("activo").count().orderBy(F.desc("count")).show(truncate=False)

### Hallazgos para la Etapa 2 (limpieza)
- **Fechas** de boleta en cinco formatos (`dd/mm/aaaa`, `aaaa-mm-dd`, `dd-mm-aa`, `d-mmm-aaaa`, `dd-mmm-aaaa`), más
  boletas con año 2026 que no calzan con la columna `mes`.
- **Duplicados** exactos en boletas. En el detalle hay pares (`boleta_id`, `linea`) repetidos: se revisa si son
  copias o productos distintos con el mismo número de línea.
- **Montos** como texto con punto de miles; **descuentos** escritos como `10%`, `0,10` y `10`; montos negativos que se
  interpretan (devoluciones).
- **Maestro de productos** con SKUs repetidos, categorías escritas de varias formas, precios con formatos mezclados,
  `N/A` en costos, `activo` escrito de seis formas (`Sí`, `SI`, `si`, `1`, `No`…), una fila de encabezado repetida
  a mitad del archivo y una fila `TOTAL` al final.
- **Promociones** con `descuento_pct` en dos formatos y SKUs que no existen en el maestro.
- **IPC** como variación mensual, no como índice.

La Etapa 2 lee desde `raw/{tabla}/{ID_FECHA}/` y deja las tablas limpias (capa plata).

---
# Etapa 2 - Limpieza (capa plata)

La Etapa 2 lee desde la zona raw y deja tablas limpias y tipadas en `plata/`. Desde aquí se trabaja con **pandas**: el
volumen (unas 15 mil líneas) no necesita Spark, y la lógica queda más legible para revisarla.

**Cada regla queda registrada.** La función `registrar()` anota cada decisión con su justificación, el origen de la
evidencia (verbalizado, medido o inferido), cuántas filas toca, qué pasaría si el supuesto
fuera falso y quién lo decidió. Al final se exporta como `salidas/bitacora_supuestos_limpieza.csv`, que es la base del anexo
 *Bitácora de supuestos*. También se exporta la **cobertura** (`salidas/cobertura.csv`): cuántos registros
quedaron fuera y por qué.

Fuentes de las decisiones: acta de la reunión del 4 de septiembre, correo de Marketing de Multitienda del Sur (respuesta a
la consulta del grupo sobre la tarjeta y los descuentos) y mediciones sobre los datos.

| Sección | Tabla plata |
|---|---|
| 2.1 Maestro de productos | `productos` |
| 2.2 Promociones | `campanas`, `campana_sku` |
| 2.3 IPC | `ipc_mensual` |
| 2.4 Tiendas, comunas, regiones y medios de pago | `tiendas`, `comunas`, `regiones`, `medios_pago` |
| 2.5 Clientes | `clientes` |
| 2.6 Boletas | `boletas` |
| 2.7 Detalle de boletas | `detalle` |
| 2.8 Cobertura y bitácora | `salidas/` |

In [ ]:
import re, unicodedata
import numpy as np

PLATA, SALIDAS = "plata", "salidas"
os.makedirs(PLATA, exist_ok=True)
os.makedirs(SALIDAS, exist_ok=True)
DECIDIO = "Equipo Grupo 3, 12-oct-2026"

def leer_raw(tabla):
    return pd.read_parquet(f"{RAW}/{tabla}/{ID_FECHA}").drop(columns=["_fuente", "_id_fecha_carga"])

def guardar_plata(df, tabla):
    df.to_parquet(f"{PLATA}/{tabla}.parquet", index=False)
    print(f"  -> plata/{tabla}.parquet ({len(df):,} filas)")

bitacora_supuestos = []
def registrar(codigo, tabla, regla, justificacion, filas, si_fuera_falso, origen):
    bitacora_supuestos.append({
        "id": codigo, "tabla": tabla, "supuesto o regla": regla, "justificación": justificacion,
        "origen de la evidencia": origen, "filas afectadas": int(filas),
        "qué pasa con el resultado si es falso": si_fuera_falso, "decidió": DECIDIO})
    print(f"[{codigo}] {regla}: {int(filas):,} filas")

# Funciones de normalización
def a_numero(serie):
    # Montos escritos como texto: '1.223,00' -> 1223 ; '$1.213' -> 1213 ; '-3.231' -> -3231 ; 'N/A' -> NaN.
    # El punto es separador de miles y la coma, decimal (se verificó que ningún monto usa punto decimal).
    s = serie.astype("string").str.strip().str.replace(r"[$\s]", "", regex=True)
    s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
         "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12}

def parsear_fecha(txt):
    # Los cinco formatos del POS: aaaa-mm-dd | dd/mm/aaaa | dd-mm-aa | d-mmm-aaaa | dd-mmm-aaaa
    if txt is None or pd.isna(txt):
        return pd.NaT
    s = str(txt).strip().lower()
    m = re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})", s)
    if m:
        a, me, d = m.groups()
    else:
        m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
        if m:
            d, me, a = m.groups()
        else:
            m = re.fullmatch(r"(\d{1,2})-(\d{1,2})-(\d{2})", s)
            if m:
                d, me, a = m.groups()
                a = "20" + a
            else:
                m = re.fullmatch(r"(\d{1,2})-([a-z]{3})-(\d{4})", s)
                if not m:
                    return pd.NaT
                d, me, a = m.groups()
                me = MESES.get(me)
    try:
        return pd.Timestamp(int(a), int(me), int(d))
    except (TypeError, ValueError):
        return pd.NaT

def sin_tilde(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()

print("Funciones de limpieza listas")

## 2.1 Maestro de productos
Problemas detectados en la Etapa 1: una fila de encabezado repetida y una fila `TOTAL`, el SKU-0001 dos veces, categorías
escritas de 24 formas, precios y costos en formatos mezclados con `N/A`, y `activo` escrito de seis formas.

La normalización de categorías se valida con la subcategoría: después de normalizar, cada subcategoría debe pertenecer a
una sola categoría. `ASEO` se unifica con `Limpieza` porque ambas contienen exactamente las mismas subcategorías
(Aseo baño y Detergentes).

In [ ]:
prod_raw = leer_raw("productos")
prod = prod_raw.copy()
for c in prod.columns:
    prod[c] = prod[c].astype("string").str.strip()

es_sku = prod.sku.str.fullmatch(r"SKU-\d+", na=False)
registrar("P01", "productos", "Se eliminan filas que no son productos (encabezado repetido y fila TOTAL)",
          "No tienen un SKU válido; son artefactos de la planilla", (~es_sku).sum(),
          "Ninguno: no corresponden a productos", "medido")
prod = prod[es_sku].copy()

MAPA_CATEGORIA = {
    "abarr": "Abarrotes", "abarrotes": "Abarrotes",
    "aseo": "Limpieza", "limpieza": "Limpieza",
    "beb": "Bebidas", "bebidas": "Bebidas",
    "carn": "Carnes", "carnes": "Carnes",
    "lact": "Lácteos", "lacteos": "Lácteos",
    "pan": "Panadería", "panaderia": "Panadería",
    "perf": "Perfumería", "perfumeria": "Perfumería",
    "snk": "Snacks", "snacks": "Snacks",
}
clave = prod.categoria.map(sin_tilde)
sin_mapa = sorted(set(clave) - set(MAPA_CATEGORIA))
assert not sin_mapa, f"Categorías sin mapear: {sin_mapa}"
cambiadas = (prod.categoria != clave.map(MAPA_CATEGORIA)).sum()
prod["categoria"] = clave.map(MAPA_CATEGORIA)
assert prod.groupby("subcategoria").categoria.nunique().max() == 1, "Una subcategoría quedó en dos categorías"
registrar("P02", "productos", "Categorías unificadas a 8 nombres canónicos (ASEO = Limpieza)",
          "Mismo concepto escrito de varias formas; se valida que cada subcategoría quede en una sola categoría",
          cambiadas, "Cortes por categoría sumarían mal; no afecta la retención", "medido")

prod["nombre"] = prod.nombre.str.replace(r"\s+", " ", regex=True)
prod["precio_lista"] = a_numero(prod.precio_lista)
prod["costo_unitario"] = a_numero(prod.costo_unitario)
registrar("P03", "productos", "Precio y costo convertidos a número; 'N/A' queda como vacío",
          "Formatos 1.223,00 / $1.213 / 1.921 representan el mismo tipo de valor (pesos)",
          prod.costo_unitario.isna().sum() + prod.precio_lista.isna().sum(),
          "Solo afecta márgenes, que no se usan en la P1", "medido")

MAPA_ACTIVO = {"si": True, "1": True, "no": False}
prod["activo"] = prod.activo.map(lambda v: MAPA_ACTIVO.get(sin_tilde(v)) if pd.notna(v) else pd.NA).astype("boolean")

n_antes = len(prod)
prod = (prod.assign(_orden=prod.activo.fillna(False).map({True: 0, False: 1}))
            .sort_values(["sku", "_orden"], kind="stable")
            .drop_duplicates("sku", keep="first").drop(columns="_orden"))
registrar("P04", "productos", "SKU repetido: se conserva la versión activa",
          "El maestro debe tener un registro por SKU; la versión inactiva es la desactualizada",
          n_antes - len(prod), "Cambia el precio de lista del SKU-0001 en $33; sin efecto en la P1", "medido")

assert prod.sku.is_unique
guardar_plata(prod, "productos")
display(prod.groupby("categoria").size().rename("skus").to_frame().T)

## 2.2 Campañas de promoción
`descuento_pct` viene como `"25%"` y como `50`. Se normaliza a proporción (0,25 y 0,50). Según el correo de Marketing,
el 2x1 de Otoño Lácteos **no queda registrado** como 50% en la línea de la boleta, así que las campañas se asignan a las
ventas cruzando SKU, fechas y canal (sección 2.7), no por el descuento de la línea.

In [ ]:
camp = leer_raw("promo_campanas")
def pct_a_proporcion(v):
    s = str(v).strip().replace("%", "").replace(",", ".")
    x = float(s)
    return x / 100 if x >= 1 else x
camp["descuento_pct"] = camp.descuento_pct.map(pct_a_proporcion)
camp["fecha_ini"] = pd.to_datetime(camp.fecha_ini)
camp["fecha_fin"] = pd.to_datetime(camp.fecha_fin)
camp["canales"] = camp.canales.map(list)
registrar("C01", "campanas", "descuento_pct normalizado a proporción (25% -> 0,25; 50 -> 0,50)",
          "Dos escrituras del mismo campo en el JSON", len(camp), "Ninguno para la P1", "medido")

csku = leer_raw("promo_campana_sku")
csku["sku"] = csku.sku.str.strip()
csku["sku_en_maestro"] = csku.sku.isin(prod.sku)
registrar("C02", "campana_sku", "SKUs en promoción que no existen en el maestro se conservan marcados",
          "SKU-9001 y SKU-9007 no están en el catálogo; no se inventa su categoría",
          (~csku.sku_en_maestro).sum(), "Ninguno: no aparecen en las ventas de la campaña", "medido")

guardar_plata(camp, "campanas")
guardar_plata(csku, "campana_sku")
display(camp[["campana_id", "nombre", "tipo", "fecha_ini", "fecha_fin", "descuento_pct", "canales", "n_skus"]])

## 2.3 IPC mensual
La API entrega la **variación mensual** en porcentaje. Se construye un índice encadenado con base diciembre 2024 = 100 y
un factor para expresar montos en **pesos de diciembre de 2025**. En la P1 se usa solo para traducir los hallazgos a pesos
comparables, mientras que la deflactación completa (P4) queda fuera del alcance. Como se mencionó, las fechas de la API vienen en UTC y se convierten a
hora de Chile antes de asignar el mes.

In [ ]:
ipc = leer_raw("ipc_mensual")
ipc["mes"] = (pd.to_datetime(ipc.fecha, utc=True).dt.tz_convert("America/Santiago")
              .dt.tz_localize(None).dt.to_period("M").dt.to_timestamp())
ipc = ipc.sort_values("mes").reset_index(drop=True)
assert ipc.mes.is_unique and len(ipc) == 12, "Se esperaban 12 meses distintos de 2025"
ipc["var_mensual"] = ipc.valor / 100
ipc["indice_dic2024_100"] = 100 * (1 + ipc.var_mensual).cumprod()
ipc["factor_a_pesos_dic2025"] = ipc.indice_dic2024_100.iloc[-1] / ipc.indice_dic2024_100
ipc = ipc[["mes", "var_mensual", "indice_dic2024_100", "factor_a_pesos_dic2025"]]

inflacion_2025 = ipc.indice_dic2024_100.iloc[-1] / 100 - 1
registrar("I01", "ipc_mensual", "Índice encadenado desde las variaciones mensuales (base dic-2024 = 100)",
          "La API entrega variación mensual, no nivel; encadenar es la forma estándar de obtener el índice",
          len(ipc), "Los montos en pesos reales cambiarían, no la cantidad de clientes retenidos", "medido")
print(f"Inflación acumulada 2025 según la API: {inflacion_2025:.2%} (la empresa usa un 3% fijo)")
guardar_plata(ipc, "ipc_mensual")
display(ipc.round({"var_mensual": 4, "indice_dic2024_100": 2, "factor_a_pesos_dic2025": 4}))

## 2.4 Tiendas y medios de pago
Se arma la tabla de tiendas con comuna, región y **macrozona** según el README del caso: Metropolitana; Centro sur
(Valparaíso y Biobío); Sur (Araucanía y Los Lagos). La Tienda Online queda como un local más, con formato `Online`.

Antes se revisa la integridad de las llaves: una comuna que apunta a una región que no existe dejaría a su tienda y a sus
clientes sin región en todos los cortes.

In [ ]:
MACROZONA = {"Metropolitana": "Metropolitana", "Valparaíso": "Centro sur", "Biobío": "Centro sur",
             "Araucanía": "Sur", "Los Lagos": "Sur"}
regiones = leer_raw("pos_regiones")
comunas = leer_raw("pos_comunas")

# Huérfana: una comuna apunta a una región que no existe en la tabla de regiones
huerfanas = comunas[~comunas.region_id.isin(regiones.region_id)]
print("Comunas con región inexistente:")
display(huerfanas)
b_tmp = leer_raw("pos_boletas").drop_duplicates()
CORRECCION_REGION = {"La Florida": "1"} # La Florida es comuna de la Región Metropolitana
for comuna, region_ok in CORRECCION_REGION.items():
    tienda_comuna = leer_raw("pos_tiendas").merge(comunas, on="comuna_id").query("comuna == @comuna").tienda_id
    print(f"  Región registrada en las boletas de la tienda de {comuna}:",
          b_tmp[b_tmp.tienda_id.isin(tienda_comuna)].region.value_counts(normalize=True).round(3).to_dict())
    comunas.loc[comunas.comuna == comuna, "region_id"] = region_ok
assert comunas.region_id.isin(regiones.region_id).all(), "Quedan comunas con región inexistente"
registrar("T00", "comunas", "La Florida apuntaba a una región inexistente (region_id 9); se asigna a Metropolitana",
          "La Florida es comuna de Santiago y el 92% de las boletas de su tienda dicen Metropolitana",
          len(huerfanas), "Sin corrección, la tienda Express La Florida y sus 111 clientes quedarían sin región",
          "medido")

tiendas = (leer_raw("pos_tiendas")
           .merge(comunas, on="comuna_id", how="left")
           .merge(regiones, on="region_id", how="left"))
tiendas["macrozona"] = tiendas.region.map(MACROZONA)
for c in ["tienda_id", "comuna_id", "region_id"]:
    tiendas[c] = tiendas[c].astype(int)
assert tiendas.macrozona.notna().all()
registrar("T01", "tiendas", "Macrozona asignada según el README del caso",
          "Es la agrupación que usa la empresa para describir sus locales", len(tiendas),
          "Solo cambia la agrupación regional de los cortes", "verbalizado (README del caso)")

medios = leer_raw("pos_medios_pago")
medios["medio_id"] = medios.medio_id.astype(int)

comunas["comuna_id"] = comunas.comuna_id.astype(int)
comunas["region_id"] = comunas.region_id.astype(int)
regiones["region_id"] = regiones.region_id.astype(int)
regiones["macrozona"] = regiones.region.map(MACROZONA)

guardar_plata(tiendas, "tiendas")
guardar_plata(comunas, "comunas")
guardar_plata(regiones, "regiones")
guardar_plata(medios, "medios_pago")
display(tiendas[["tienda_id", "nombre", "formato", "comuna", "region", "macrozona"]])

## 2.5 Clientes
- `fecha_alta` viene en dos formatos y se convierte a fecha. **No se usa como fecha de primera compra**: en los datos, la
  mayoría de los clientes tiene el alta *antes* de su primera boleta de 2025 y una parte la tiene *después*. Cómo se usa
  cada fecha se decide en el modelo y la métrica.
- El RUT viene en más de 30 formatos. Se normaliza a `12345678-K` y se valida el dígito verificador. No se usa en la P1;
  se deja limpio para la dimensión cliente.
- **Segmento** (Clásico, Plus, Black): según el correo de Marketing, no tiene regla escrita, se sobrescribe sin historial
  y los tres compran parecido. Se conserva como atributo **nominal** con el estado actual; no se interpreta como escala.

In [ ]:
cli = leer_raw("pos_clientes")
cli["cliente_id"] = cli.cliente_id.astype(int)
cli["comuna_id"] = cli.comuna_id.astype(int)
cli["nombre"] = cli.nombre.str.strip()
cli["fecha_alta"] = cli.fecha_alta.map(parsear_fecha)
assert cli.fecha_alta.notna().all(), "Hay fechas de alta sin interpretar"

def normalizar_rut(txt):
    s = re.sub(r"[^0-9kK]", "", str(txt)).upper()
    if len(s) < 2:
        return pd.NA, False
    cuerpo, dv = s[:-1], s[-1]
    suma, mult = 0, 2
    for d in reversed(cuerpo):
        suma += int(d) * mult
        mult = 2 if mult == 7 else mult + 1
    resto = 11 - suma % 11
    esperado = "0" if resto == 11 else "K" if resto == 10 else str(resto)
    return f"{cuerpo}-{dv}", dv == esperado

ruts = cli.rut.map(normalizar_rut)
cli["rut"] = ruts.map(lambda t: t[0])
cli["rut_valido"] = ruts.map(lambda t: t[1])
registrar("CL01", "clientes", "RUT normalizado a 12345678-K y dígito verificador validado",
          "Más de 30 formatos para el mismo dato", (~cli.rut_valido).sum(),
          "Ninguno para la P1: el cliente se identifica por cliente_id, no por RUT", "medido")
assert cli.rut.is_unique, "Hay RUT repetidos entre clientes distintos"

registrar("CL02", "clientes", "Segmento se usa como atributo nominal con el estado actual (sin historial)",
          "Correo de Marketing: no hay regla ni beneficios por segmento, se sobrescribe y los tres compran parecido",
          len(cli), "Si el segmento ordenara el valor del cliente, el corte por segmento ocultaría esa diferencia",
          "verbalizado (correo de Marketing) y medido")

guardar_plata(cli, "clientes")
display(cli.head())

## 2.6 Boletas (cabecera)
Reglas, en orden:
1. **Duplicados exactos** (todas las columnas iguales): se elimina la copia.
2. **Fecha**: se interpretan los cinco formatos. `dd/mm/aaaa` se lee como día/mes; se verifica que
   ningún valor contradiga esa lectura.
3. **Período**: el análisis cubre 2025. Las boletas con año 2026 se excluyen (columna `mes` tampoco calza con esas
   fechas, así que corregirlas sería adivinar).
4. **Columna `mes`**: se descarta; la fuente de verdad es `fecha`.
5. **Columna `region`**: se descarta; la región sale de la tienda, porque la de la boleta no coincide con la del local en
   cientos de casos (incluidas compras presenciales).
6. **Boletas anónimas** (sin `cliente_id`): se conservan para los totales y la cobertura, y quedan fuera de la retención.
   Según el correo de Marketing, son las compras de quien no está en el programa, no una cuarta categoría.

In [ ]:
bol_raw = leer_raw("pos_boletas")
n_origen_boletas = len(bol_raw)

bol = bol_raw.drop_duplicates().copy()
registrar("B01", "boletas", "Se elimina la copia de las boletas duplicadas exactas",
          "Misma boleta_id y mismos valores en todas las columnas: es una carga doble",
          n_origen_boletas - len(bol), "Se sobrecontarían 62 visitas", "medido")
assert bol.boleta_id.is_unique, "Quedan boleta_id repetidos con contenido distinto"

bol["fecha_origen"] = bol.fecha
dmy = bol.fecha.str.strip().str.extract(r"^(\d{1,2})/(\d{1,2})/\d{4}$").dropna().astype(int)
print(f"Formato dd/mm/aaaa: {(dmy[0] > 12).sum()} valores solo posibles como día/mes, "
      f"{(dmy[1] > 12).sum()} que contradirían esa lectura")
assert (dmy[1] > 12).sum() == 0
bol["fecha"] = bol.fecha.map(parsear_fecha)
assert bol.fecha.notna().all(), "Hay fechas sin interpretar"
registrar("B02", "boletas", "Fechas unificadas desde 5 formatos; dd/mm/aaaa leído como día/mes",
          "Formato chileno; ningún valor tiene un mes mayor que 12 en la segunda posición", len(bol),
          "Si fuera mes/día, se moverían de mes boletas con día <= 12", "medido")

fuera = bol.fecha.dt.year != 2025
registrar("B03", "boletas", "Se excluyen boletas con fecha fuera de 2025 (año 2026)",
          "Los datos del caso son de 2025; la columna mes tampoco calza con esas fechas",
          fuera.sum(), "Si fueran ventas de 2025 mal digitadas, faltaría un 0,6% de las visitas", "medido")
bol = bol[~fuera].copy()

mes_distinto = bol.mes.astype(str).str.strip() != bol.fecha.dt.month.astype(str)
registrar("B04", "boletas", "Se descarta la columna mes; manda la fecha",
          "Contradice a la fecha en cientos de casos; la fecha es el dato primario", mes_distinto.sum(),
          "Si mes fuera la correcta, cambiaría el mes asignado a esas boletas", "medido")

bol["tienda_id"] = bol.tienda_id.astype(int)
bol["medio_id"] = bol.medio_id.astype(int)
reg_tienda = bol.tienda_id.map(tiendas.set_index("tienda_id").region)
registrar("B05", "boletas", "Se descarta la columna region de la boleta; la región sale de la tienda",
          "No coincide con la región del local en compras presenciales, donde la región es la del local",
          (bol.region != reg_tienda).sum(), "Cambiaría la región de esas boletas en los cortes regionales", "medido")

bol["cliente_id"] = pd.to_numeric(bol.cliente_id).astype("Int64")
bol["es_anonima"] = bol.cliente_id.isna()
registrar("B06", "boletas", "Boletas anónimas: se conservan en totales y fuera de la retención",
          "Sin cliente_id no se puede seguir a nadie en el tiempo; correo de Marketing: son compras fuera del programa",
          bol.es_anonima.sum(), "Si fueran clientes frecuentes sin tarjeta, la retención real sería mayor que la medida",
          "verbalizado (acta y correo) y medido")

bol["boleta_id"] = bol.boleta_id.astype(int)
bol = bol[["boleta_id", "fecha", "fecha_origen", "tienda_id", "cliente_id", "es_anonima", "canal", "medio_id"]]
print(f"\nBoletas 2025 válidas: {len(bol):,} | anónimas: {bol.es_anonima.mean():.1%}")

## 2.7 Detalle de boletas (líneas)
1. Solo se conservan las líneas de boletas válidas.
2. **Número de línea repetido**: hay pares (`boleta_id`, `linea`) repetidos, pero son **productos distintos** (otro SKU,
   otro precio), no copias. Se conservan todos y se crea una llave nueva `linea_id`; el número original queda en
   `linea_origen`.
3. Precio y monto a número; **descuento** normalizado: `10%`, `10` y `0,10` son el mismo 10% (correo de Marketing).
4. **Devoluciones**: líneas con cantidad negativa. Restan del monto de la boleta.
5. **Cuadre**: si `monto_linea` no es igual a precio × cantidad × (1 − descuento) (tolerancia de $1), la línea **no se
   corrige** y queda marcada `linea_cuadra = False`, como pidió Marketing. El ingreso usa `monto_linea`, que es lo cobrado.
6. **Campaña**: una línea pertenece a una campaña si su SKU está en la campaña, la fecha cae en el rango y el canal está
   en la lista de canales de la campaña.
7. **SKU sin maestro**: se conservan en las ventas, marcados, sin atributos de producto.

In [ ]:
det_raw = leer_raw("pos_boleta_detalle")
det = det_raw.copy()
det["boleta_id"] = det.boleta_id.astype(int)
det = det[det.boleta_id.isin(bol.boleta_id)].copy()
registrar("D01", "detalle", "Se excluyen las líneas de boletas excluidas (fuera de 2025)",
          "Siguen a su boleta", len(det_raw) - len(det), "Igual que B03", "medido")

det["linea_origen"] = det.linea.astype(int)
repetidas = det.duplicated(["boleta_id", "linea_origen"], keep=False)
iguales = det.duplicated(keep=False)
det = det.sort_values(["boleta_id", "linea_origen"], kind="stable")
det["linea_id"] = det.groupby("boleta_id").cumcount() + 1
registrar("D02", "detalle", "Número de línea repetido: se conservan las líneas y se crea linea_id",
          "Las líneas con número repetido tienen otro SKU y otro precio: son productos distintos, no copias",
          det.duplicated(["boleta_id", "linea_origen"]).sum(),
          "Si fueran copias, se sobrecontaría ingreso (no visitas ni clientes)", "medido")
assert not (repetidas & iguales).any(), "Hay líneas que sí son copias exactas"

det["sku"] = det.sku.str.strip()
det["cantidad"] = det.cantidad.astype(int)
det["precio_unitario"] = a_numero(det.precio_unitario)
det["monto_linea"] = a_numero(det.monto_linea)
assert det[["precio_unitario", "monto_linea"]].notna().all().all()

def normalizar_descuento(txt):
    s = str(txt).strip().replace(",", ".")
    v = float(s[:-1]) / 100 if s.endswith("%") else float(s)
    return round(v / 100 if v >= 1 else v, 4)
det["descuento"] = det.descuento.map(normalizar_descuento)
assert set(det.descuento) <= {0.0, 0.10, 0.25}, set(det.descuento)
registrar("D03", "detalle", "Descuento normalizado a 0 / 0,10 / 0,25",
          "Correo de Marketing: 10%, 10 y 0,10 son el mismo 10%", (det.descuento > 0).sum(),
          "Solo afecta montos esperados y el cuadre", "verbalizado (correo de Marketing)")

det["es_devolucion"] = det.cantidad < 0
registrar("D04", "detalle", "Cantidad negativa = devolución; resta del monto de la boleta",
          "Todas tienen monto negativo y cuadran con precio x cantidad", det.es_devolucion.sum(),
          "Si no fueran devoluciones, el ingreso neto estaría subestimado", "inferido")

det["monto_esperado"] = (det.precio_unitario * det.cantidad * (1 - det.descuento)).round(0)
det["linea_cuadra"] = (det.monto_linea - det.monto_esperado).abs() <= 1
registrar("D05", "detalle", "Líneas que no cuadran: no se corrigen, se marcan y se reportan",
          "Correo de Marketing: no forzar; el ingreso usa monto_linea (lo cobrado)", (~det.linea_cuadra).sum(),
          "Si el monto correcto fuera el esperado, el ingreso bajaría; la retención (conteo de clientes) no cambia",
          "verbalizado (correo de Marketing) y medido")

# Campañas: SKU + rango de fechas + canal
x = (det[["boleta_id", "linea_id", "sku"]]
     .merge(bol[["boleta_id", "fecha", "canal"]], on="boleta_id")
     .merge(csku[["campana_id", "sku"]], on="sku")
     .merge(camp[["campana_id", "fecha_ini", "fecha_fin", "canales"]], on="campana_id"))
x = x[(x.fecha >= x.fecha_ini.dt.normalize()) & (x.fecha <= x.fecha_fin)
      & x.apply(lambda r: r.canal in r.canales, axis=1)]
assert not x.duplicated(["boleta_id", "linea_id"]).any(), "Una línea cae en dos campañas"
det = det.merge(x[["boleta_id", "linea_id", "campana_id"]], on=["boleta_id", "linea_id"], how="left")
registrar("D06", "detalle", "Campaña asignada por SKU + fecha + canal, no por el descuento de la línea",
          "Correo de Marketing: el 2x1 no queda guardado como 50% en la línea", det.campana_id.notna().sum(),
          "Solo afecta el análisis de promociones (P3, fuera de alcance)", "verbalizado (correo de Marketing)")

sep = det.campana_id.eq("CAMP-2025-09")
print(f"  Septiembre: {(sep & (det.descuento == 0.25)).sum()} líneas de la campaña con 25%, "
      f"{(sep & (det.descuento != 0.25)).sum()} sin él; {(~sep & (det.descuento == 0.25)).sum()} líneas con 25% fuera de la campaña")
print(f"  10%: {(det.descuento == 0.10).sum():,} líneas; en boletas anónimas: "
      f"{det[det.descuento == 0.10].boleta_id.isin(bol[bol.es_anonima].boleta_id).sum()}")

det["sku_en_maestro"] = det.sku.isin(prod.sku)
registrar("D07", "detalle", "Líneas con SKU que no está en el maestro: se conservan, sin atributos de producto",
          "Son ventas reales; el catálogo no está homologado (acta)", (~det.sku_en_maestro).sum(),
          "Ninguno para la P1; los cortes por categoría quedan con un grupo 'Sin maestro'",
          "verbalizado (acta) y medido")

det = det[["boleta_id", "linea_id", "linea_origen", "sku", "cantidad", "precio_unitario", "descuento",
           "monto_linea", "monto_esperado", "linea_cuadra", "es_devolucion", "campana_id", "sku_en_maestro"]]

### Monto de la boleta y definición de visita
El monto de cada boleta es la suma de sus `monto_linea` (neto de devoluciones). Una boleta cuenta como **visita** solo
si su monto es positivo (una devolución sola no debe hacer parecer activo a un cliente).

In [ ]:
resumen = det.groupby("boleta_id").agg(monto_boleta=("monto_linea", "sum"), n_lineas=("linea_id", "size"),
                                       tiene_devolucion=("es_devolucion", "any"))
bol = bol.merge(resumen, on="boleta_id", how="left")
assert bol.monto_boleta.notna().all(), "Hay boletas sin líneas"
bol["es_visita"] = bol.monto_boleta > 0
registrar("B07", "boletas", "Una boleta es visita solo si su monto neto es positivo",
          "Una devolución sin compra no es una visita de compra", (~bol.es_visita).sum(),
          "Si contaran, algunos clientes parecerían activos sin haber comprado", "inferido")

guardar_plata(bol, "boletas")
guardar_plata(det, "detalle")

## 2.8 Cobertura de datos y bitácora de supuestos
La cobertura responde: **¿qué porcentaje de los registros quedó fuera del análisis de retención y por qué?** La
retención solo puede medirse sobre visitas de clientes identificados.

In [ ]:
n_dup = n_origen_boletas - len(bol_raw.drop_duplicates())
n_fuera = int(fuera.sum())
n_validas = len(bol)
n_anon = int(bol.es_anonima.sum())
n_ident = n_validas - n_anon
n_no_visita_ident = int((~bol.es_anonima & ~bol.es_visita).sum())
n_usadas = n_ident - n_no_visita_ident

pasos = [
    ("Boletas en origen", n_origen_boletas, ""),
    ("(-) Duplicadas exactas", -n_dup, "B01"),
    ("(-) Fuera de 2025", -n_fuera, "B03"),
    ("= Boletas válidas 2025", n_validas, ""),
    ("(-) Anónimas (sin cliente_id)", -n_anon, "B06"),
    ("(-) Identificadas sin monto positivo", -n_no_visita_ident, "B07"),
    ("= Visitas identificadas usadas en la retención", n_usadas, ""),
]
cobertura = pd.DataFrame(pasos, columns=["paso", "boletas", "regla"])
cobertura["% del origen"] = (cobertura.boletas.abs() / n_origen_boletas * 100).round(1)
cobertura.to_csv(f"{SALIDAS}/cobertura.csv", index=False)
display(cobertura)

monto_total = bol.monto_boleta.sum()
monto_ident = bol.loc[~bol.es_anonima, "monto_boleta"].sum()
print(f"Boletas fuera del análisis de retención: {(n_origen_boletas - n_usadas) / n_origen_boletas:.1%} del origen")
print(f"Ingreso 2025 cubierto por clientes identificados: {monto_ident / monto_total:.1%} "
      f"(${monto_ident:,.0f} de ${monto_total:,.0f})")
print(f"Líneas que no cuadran (se conservan): {(~det.linea_cuadra).mean():.1%} de las líneas válidas")

clientes_sin_compra = (~cli.cliente_id.isin(bol.loc[bol.es_visita, "cliente_id"].dropna())).sum()
print(f"Clientes del maestro sin ninguna visita en 2025: {clientes_sin_compra}")

In [ ]:
bitacora_df = pd.DataFrame(bitacora_supuestos)
bitacora_df.to_csv(f"{SALIDAS}/bitacora_supuestos_limpieza.csv", index=False, encoding="utf-8-sig")
with pd.option_context("display.max_colwidth", 80):
    display(bitacora_df[["id", "supuesto o regla", "origen de la evidencia", "filas afectadas"]])
print(f"Bitácora exportada: {SALIDAS}/bitacora_supuestos_limpieza.csv ({len(bitacora_df)} reglas)")

### Cierre de la Etapa 2
Quedan en `plata/` las tablas limpias que alimentan el modelo dimensional (Etapa 3): `boletas`, `detalle`, `clientes`,
`tiendas`, `comunas`, `regiones`, `medios_pago`, `productos`, `campanas`, `campana_sku` e `ipc_mensual`. En `salidas/` quedan la cobertura y
la bitácora de supuestoS (anexo informe).

---
# Etapa 3 - Modelo dimensional (capa oro)

Esquema en estrella construido desde `plata/` y guardado en `oro/`.

**Grano de la tabla de hechos `fact_venta_linea`:**

> **Una fila es un producto (SKU) vendido o devuelto en una boleta válida de 2025.**

| Tabla | Rol | Origen |
|---|---|---|
| `fact_venta_linea` | Hechos al grano atómico | POS (detalle + cabecera) |
| `dim_fecha` | Calendario 2025 con IPC del mes | API mindicador.cl |
| `dim_cliente` | Cliente del programa + miembro "Anónimo" | POS (clientes, comunas, regiones) |
| `dim_tienda` | Local, formato, canal, región, macrozona | POS (tiendas, comunas, regiones, boletas) |
| `dim_producto` | SKU con categoría, marca, proveedor | Maestro de productos (CSV) |
| `dim_campana` | Campaña + miembro "Sin campaña" | Promociones (JSON) |

Las cuatro fuentes entran al modelo: el POS alimenta los hechos y dos dimensiones, el CSV a `dim_producto`, el JSON a
`dim_campana` y la API a `dim_fecha`.

### Decisiones de diseño
1. **Grano atómico (línea) y no boleta.** La retención se mide en visitas (boletas), pero si el hecho fuera la boleta, el
   producto y la campaña no podrían colgar de ella: una boleta tiene muchos productos. Con el grano de línea, el CSV y el
   JSON se integran de verdad. Las visitas se obtienen agregando las líneas por `boleta_id` con una sola función
   (`visitas_por_boleta`), que todos los cálculos de la métrica reutilizan; así no hace falta una segunda tabla de hechos
   y la definición de visita es siempre la misma.
2. **La llave del hecho no es (`boleta_id`, `linea`).** Ese par se repite en 408 líneas que son productos distintos
   (Etapa 2, regla D02). La llave es (`boleta_id`, `linea_id`), con `linea_id` creado en la limpieza.
3. **Miembros especiales en lugar de nulos.** Las boletas anónimas apuntan a `cliente_key = 0` ("Anónimo"), las líneas sin
   campaña a `campana_key = 0` ("Sin campaña"), y cada SKU que no está en el maestro obtiene su propia fila en
   `dim_producto` con categoría "Sin maestro". Así ninguna venta se pierde en un *join* y los totales siempre cuadran.
4. **`dim_cliente` es tipo 1 (sin historial).** El segmento se sobrescribe en origen y no hay forma de reconstruir su
   historia (correo de Marketing). Se guarda el estado actual y se documenta como límite.
5. **Tipo de cliente según la fecha de alta.** `Base heredada` si se inscribió antes de 2025; `Nuevo 2025` si se inscribió
   durante 2025. Sin esta separación, todo cliente antiguo que compra en enero parecería "nuevo" y se inflaría la
   reposición, que es justo lo que la pregunta quiere medir.
6. **El IPC vive en `dim_fecha`.** Cada día lleva la variación, el índice y el factor a pesos de diciembre de 2025 de su
   mes, lo que permite expresar cualquier monto en pesos reales sin otra tabla.
7. **El canal es un atributo de la tienda, no una dimensión propia.** En los datos, toda boleta Web es de la Tienda Online
   y toda boleta de la Tienda Online es Web: el canal queda determinado por la tienda. Una `dim_canal` separada repetiría
   la misma información con otra llave. Se verifica esta regla antes de construir el modelo.
8. **El medio de pago queda fuera del modelo.** Se midió y no distingue comportamiento: cada medio pesa cerca de un 25% en
   ambos canales, con la misma proporción de anónimas y un ticket parecido, y la mayoría de los clientes usa más de uno.
   No aporta a la pregunta de retención, y el enunciado pide un modelo que sostenga la pregunta, no el caso entero.

In [ ]:
ORO = "oro"
os.makedirs(ORO, exist_ok=True)

def leer_plata(t):
    return pd.read_parquet(f"{PLATA}/{t}.parquet")

def guardar_oro(df, t):
    df.to_parquet(f"{ORO}/{t}.parquet", index=False)
    print(f"  -> oro/{t}.parquet ({len(df):,} filas, {df.shape[1]} columnas)")

bol_p, det_p, cli_p = leer_plata("boletas"), leer_plata("detalle"), leer_plata("clientes")
tie_p, com_p, reg_p = leer_plata("tiendas"), leer_plata("comunas"), leer_plata("regiones")
med_p, prod_p, camp_p, ipc_p = leer_plata("medios_pago"), leer_plata("productos"), leer_plata("campanas"), leer_plata("ipc_mensual")

NOMBRE_MES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
              "septiembre", "octubre", "noviembre", "diciembre"]
NOMBRE_DIA = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
print("Tablas plata cargadas")

## 3.1 Dimensiones

In [ ]:
# dim_fecha: un día por fila, con el IPC de su mes
dias = pd.DataFrame({"fecha": pd.date_range("2025-01-01", "2025-12-31", freq="D")})
dias["mes"] = dias.fecha.dt.to_period("M").dt.to_timestamp()
dim_fecha = dias.merge(ipc_p, on="mes", how="left")
assert dim_fecha.factor_a_pesos_dic2025.notna().all(), "Hay días sin IPC"
dim_fecha = pd.DataFrame({
    "fecha_key": dim_fecha.fecha.dt.strftime("%Y%m%d").astype(int),
    "fecha": dim_fecha.fecha,
    "anio": dim_fecha.fecha.dt.year,
    "trimestre": "T" + dim_fecha.fecha.dt.quarter.astype(str),
    "mes_num": dim_fecha.fecha.dt.month,
    "mes_nombre": dim_fecha.fecha.dt.month.map(lambda m: NOMBRE_MES[m - 1]),
    "semana_iso": dim_fecha.fecha.dt.isocalendar().week.astype(int),
    "dia_semana": dim_fecha.fecha.dt.dayofweek.map(lambda d: NOMBRE_DIA[d]),
    "es_fin_de_semana": dim_fecha.fecha.dt.dayofweek >= 5,
    "ipc_var_mensual": dim_fecha.var_mensual,
    "ipc_indice_dic2024_100": dim_fecha.indice_dic2024_100,
    "factor_a_pesos_dic2025": dim_fecha.factor_a_pesos_dic2025,
})

# dim_tienda, con el canal como atributo (el canal está determinado por la tienda)
canal_por_tienda = bol_p.groupby("tienda_id").canal.agg(["nunique", "first"])
assert (canal_por_tienda["nunique"] == 1).all(), "Hay tiendas con boletas de más de un canal"
dim_tienda = tie_p.rename(columns={"nombre": "tienda"})[
    ["tienda_id", "tienda", "formato", "comuna", "region", "macrozona"]].copy()
dim_tienda.insert(0, "tienda_key", dim_tienda.tienda_id)
dim_tienda["canal"] = dim_tienda.tienda_id.map(canal_por_tienda["first"])
dim_tienda["es_online"] = dim_tienda.canal.eq("Web")
assert dim_tienda.es_online.eq(dim_tienda.formato.eq("Online")).all()
registrar("M04", "dim_tienda", "El canal se modela como atributo de la tienda y no como dimensión propia",
          "Toda boleta Web es de la Tienda Online y viceversa: el canal queda determinado por la tienda",
          len(dim_tienda), "Si una tienda física vendiera por web, habría que volver a una dimensión de canal",
          "medido")

# Medio de pago: solo se mide si distingue comportamiento antes de decidir si entra al modelo
mp = bol_p.merge(med_p, on="medio_id")
print("Participación de cada medio de pago por canal:")
display(pd.crosstab(mp.medio, mp.canal, normalize="columns").round(3))
display(mp.groupby("medio").agg(boletas=("boleta_id", "size"), pct_anonimas=("es_anonima", "mean"),
                                ticket_mediano=("monto_boleta", "median")).round(3))
mp_id = mp[mp.es_visita & ~mp.es_anonima]
multi_medio = (mp_id.groupby("cliente_id").medio.nunique() > 1).mean()
efectivo_web = mp[mp.canal == "Web"].medio.eq("Efectivo").mean()
print(f"Clientes que usan más de un medio: {multi_medio:.0%} | compras web pagadas en efectivo: {efectivo_web:.0%}")
registrar("M05", "modelo", "El medio de pago queda fuera del modelo dimensional",
          f"No distingue comportamiento (cada medio ~25%, misma proporción de anónimas y ticket similar; "
          f"{multi_medio:.0%} de los clientes usa más de uno); no aporta a la P1",
          len(med_p), "Si el medio de pago explicara la retención, el modelo no permitiría verlo; se puede reincorporar desde plata/",
          "medido")
print(f"Observación de calidad: {efectivo_web:.0%} de las compras web figura pagada en efectivo (¿pago contra entrega?)")

display(dim_tienda)

In [ ]:
# dim_cliente: atributos del programa + fecha de primera visita de 2025 (adquisición) + tipo de cliente.
# Sin RUT ni nombre (minimización por Ley 21.719) y sin conteos de visitas (son medidas: viven en el hecho).
visitas_id = bol_p[bol_p.es_visita & ~bol_p.es_anonima]
resumen_cli = visitas_id.groupby("cliente_id").agg(primera_visita_2025=("fecha", "min"))
geo = com_p.merge(reg_p, on="region_id")[["comuna_id", "comuna", "region", "macrozona"]]
dim_cliente = (cli_p.merge(geo, on="comuna_id", how="left")
                    .merge(resumen_cli, left_on="cliente_id", right_index=True, how="left"))
dim_cliente["tipo_cliente"] = np.where(dim_cliente.fecha_alta < "2025-01-01", "Base heredada", "Nuevo 2025")
dim_cliente["alta_posterior_a_primera_visita"] = dim_cliente.fecha_alta > dim_cliente.primera_visita_2025
dim_cliente.insert(0, "cliente_key", dim_cliente.cliente_id)
dim_cliente = dim_cliente[["cliente_key", "cliente_id", "segmento", "tipo_cliente", "fecha_alta",
                           "alta_posterior_a_primera_visita", "primera_visita_2025", "comuna", "region", "macrozona"]]

anonimo = pd.DataFrame([{"cliente_key": 0, "cliente_id": pd.NA, "segmento": "Sin programa (anónimo)",
                         "tipo_cliente": "Anónimo", "fecha_alta": pd.NaT, "alta_posterior_a_primera_visita": pd.NA,
                         "primera_visita_2025": pd.NaT, "comuna": pd.NA, "region": pd.NA, "macrozona": pd.NA}])
dim_cliente = pd.concat([anonimo.astype(dim_cliente.dtypes.to_dict(), errors="ignore"), dim_cliente],
                        ignore_index=True)
dim_cliente["cliente_id"] = dim_cliente.cliente_id.astype("Int64")

reales = dim_cliente.cliente_key > 0
registrar("M01", "dim_cliente", "Tipo de cliente según fecha de alta: antes de 2025 = Base heredada; en 2025 = Nuevo 2025",
          "Los datos solo cubren 2025: la primera boleta del año no distingue a un cliente nuevo de uno antiguo que vuelve",
          (dim_cliente[reales].tipo_cliente == "Base heredada").sum(),
          "Si la fecha de alta no reflejara la antigüedad, se mezclarían retenidos con adquiridos", "medido e inferido")
registrar("M02", "dim_cliente", "Clientes con fecha de alta posterior a su primera visita: se conservan sin corregir el alta",
          "Contraparte: el alta 'es la primera compra', con advertencia de revisarlo; los datos lo contradicen",
          int((dim_cliente[reales].alta_posterior_a_primera_visita == True).sum()),
          "Ninguno para el tipo de cliente salvo que el alta caiga en otro año que la primera visita",
          "verbalizado (acta) y medido")
registrar("M06", "dim_cliente", "dim_cliente sin RUT ni nombre y sin conteos de visitas",
          "Minimización (Ley 21.719, clase 3): la P1 se responde con cliente_id. Los conteos son medidas y van en el hecho "
          "(un monto o conteo dentro de la dimensión es un error de diseño, clase 10)", int(reales.sum()),
          "Ninguno: RUT y nombre siguen en plata/clientes si una pregunta futura los necesita", "inferido")
display(dim_cliente[reales].groupby("tipo_cliente").agg(clientes=("cliente_key", "size"),
                                                        con_visita_2025=("primera_visita_2025", lambda s: s.notna().sum())))

In [ ]:
# dim_producto: maestro + un miembro por cada SKU vendido que no está en el maestro
dim_producto = prod_p[["sku", "nombre", "categoria", "subcategoria", "marca", "proveedor",
                       "precio_lista", "costo_unitario", "activo"]].copy()
dim_producto["en_maestro"] = True
skus_sin_maestro = sorted(set(det_p.sku) - set(dim_producto.sku))
faltantes_prod = pd.DataFrame({"sku": skus_sin_maestro, "nombre": "SKU sin maestro", "categoria": "Sin maestro",
                               "subcategoria": "Sin maestro", "marca": "Sin maestro", "proveedor": "Sin maestro",
                               "en_maestro": False})
dim_producto = pd.concat([dim_producto, faltantes_prod], ignore_index=True)
dim_producto.insert(0, "producto_key", range(1, len(dim_producto) + 1))
registrar("M03", "dim_producto", "Cada SKU vendido que no está en el maestro obtiene su fila con categoría 'Sin maestro'",
          "Conserva el código del SKU y evita perder ventas en el cruce con productos", len(skus_sin_maestro),
          "Ninguno para la P1; los cortes por categoría muestran el grupo 'Sin maestro'", "medido")

# dim_campana: campañas del JSON + miembro "Sin campaña"
dim_campana = camp_p[["campana_id", "nombre", "tipo", "fecha_ini", "fecha_fin", "descuento_pct"]].copy()
dim_campana["canales"] = camp_p.canales.map(lambda c: ", ".join(c))
dim_campana.insert(0, "campana_key", range(1, len(dim_campana) + 1))
sin_camp = pd.DataFrame([{"campana_key": 0, "campana_id": "SIN", "nombre": "Sin campaña", "tipo": "Ninguna",
                          "fecha_ini": pd.NaT, "fecha_fin": pd.NaT, "descuento_pct": 0.0, "canales": ""}])
dim_campana = pd.concat([sin_camp, dim_campana], ignore_index=True)

display(dim_campana)

## 3.2 Tabla de hechos `fact_venta_linea`
Una fila por producto vendido o devuelto en una boleta válida de 2025. Medidas: cantidad, precio unitario, descuento,
monto cobrado (nominal), monto en pesos de diciembre de 2025 (con el IPC de `dim_fecha`) y monto esperado.
`boleta_id` y `linea_id` son dimensiones degeneradas: identifican la línea pero no tienen tabla propia.

In [ ]:
f = det_p.merge(bol_p[["boleta_id", "fecha", "tienda_id", "cliente_id"]], on="boleta_id", how="left")
assert f.fecha.notna().all(), "Hay líneas sin boleta"

f["fecha_key"] = f.fecha.dt.strftime("%Y%m%d").astype(int)
f["cliente_key"] = f.cliente_id.fillna(0).astype(int)
f["tienda_key"] = f.tienda_id
f["producto_key"] = f.sku.map(dim_producto.set_index("sku").producto_key)
f["campana_key"] = f.campana_id.map(dim_campana.set_index("campana_id").campana_key).fillna(0).astype(int)
f["monto_linea_real_dic2025"] = (f.monto_linea * f.fecha_key.map(dim_fecha.set_index("fecha_key").factor_a_pesos_dic2025)).round(0)

fact_venta_linea = f[["boleta_id", "linea_id", "fecha_key", "cliente_key", "tienda_key",
                      "producto_key", "campana_key", "cantidad", "precio_unitario", "descuento", "monto_linea",
                      "monto_linea_real_dic2025", "monto_esperado", "linea_cuadra", "es_devolucion"]].copy()
fact_venta_linea.head()

## 3.3 Definición de visita
La retención se mide en visitas, no en líneas. La función `visitas_por_boleta` agrega `fact_venta_linea` por `boleta_id`
(usando la dimensión degenerada) y aplica la regla B07: una boleta es visita solo si su monto neto es positivo. Todos los
cálculos de la Etapa 4 usan esta misma función, para que la definición de visita sea única.

In [ ]:
def visitas_por_boleta(hechos):
    # Una fila por boleta con su monto neto; es_visita = monto neto > 0 (regla B07)
    b = (hechos.groupby(["boleta_id", "fecha_key", "cliente_key", "tienda_key"], as_index=False)
               .agg(monto_boleta=("monto_linea", "sum"), monto_boleta_real_dic2025=("monto_linea_real_dic2025", "sum")))
    assert b.boleta_id.is_unique, "Una boleta quedó con dos fechas, tiendas o clientes"
    b["es_visita"] = b.monto_boleta > 0
    b["es_anonima"] = b.cliente_key.eq(0)
    return b

boletas_m = visitas_por_boleta(fact_venta_linea)
print(f"Boletas: {len(boletas_m):,} | visitas identificadas: {(boletas_m.es_visita & ~boletas_m.es_anonima).sum():,}")

## 3.4 Controles de integridad
El modelo se valida antes de usarlo. Si un control falla, se detiene.

In [ ]:
controles = []
def control(nombre, ok, detalle=""):
    controles.append({"control": nombre, "resultado": "OK" if ok else "FALLA", "detalle": detalle})
    assert ok, f"Falla el control: {nombre} {detalle}"

control("Grano único (boleta_id, linea_id)", not fact_venta_linea.duplicated(["boleta_id", "linea_id"]).any())
control("Mismas filas que plata/detalle", len(fact_venta_linea) == len(det_p), f"{len(fact_venta_linea):,} filas")
for dim, llave, tabla in [(dim_fecha, "fecha_key", "dim_fecha"), (dim_cliente, "cliente_key", "dim_cliente"),
                          (dim_tienda, "tienda_key", "dim_tienda"),
                          (dim_producto, "producto_key", "dim_producto"), (dim_campana, "campana_key", "dim_campana")]:
    control(f"Llave única en {tabla}", dim[llave].is_unique)
    huerfanas = (~fact_venta_linea[llave].isin(dim[llave])).sum()
    control(f"Sin llaves huérfanas hacia {tabla}", huerfanas == 0, f"{huerfanas} huérfanas")
control("Monto total igual a plata/boletas",
        abs(fact_venta_linea.monto_linea.sum() - bol_p.monto_boleta.sum()) < 1, f"${fact_venta_linea.monto_linea.sum():,.0f}")
control("Boletas agregadas igual a plata/boletas", len(boletas_m) == len(bol_p), f"{len(boletas_m):,}")
control("Visitas identificadas igual a la cobertura (Etapa 2)",
        (boletas_m.es_visita & ~boletas_m.es_anonima).sum() == n_usadas, f"{n_usadas:,}")
display(pd.DataFrame(controles))

In [ ]:
for t, df in [("dim_fecha", dim_fecha), ("dim_cliente", dim_cliente), ("dim_tienda", dim_tienda),
              ("dim_producto", dim_producto),
              ("dim_campana", dim_campana), ("fact_venta_linea", fact_venta_linea)]:
    guardar_oro(df, t)

## 3.5 Diagrama del modelo
Se genera con código para que siempre refleje las tablas reales. Se guarda en `salidas/modelo_dimensional.png`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def caja(ax, x, y, w, titulo, filas, color, fuente, punteada=False):
    alto = 0.42 + 0.30 * len(filas)
    ax.add_patch(FancyBboxPatch((x, y - alto), w, alto, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc="white", ec=color, lw=1.6, ls="--" if punteada else "-"))
    ax.add_patch(FancyBboxPatch((x, y - 0.42), w, 0.42, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=color, ec=color, lw=1.6))
    ax.text(x + 0.12, y - 0.21, titulo, color="white", fontsize=10.5, fontweight="bold", va="center")
    for i, (txt, tipo) in enumerate(filas):
        peso = "bold" if tipo == "pk" else "normal"
        col = "#6b7280" if tipo == "attr" else "#111827"
        ax.text(x + 0.14, y - 0.62 - 0.30 * i, txt, fontsize=8.6, va="center", fontweight=peso, color=col)
    if fuente:
        ax.text(x + w - 0.12, y - 0.21, fuente, fontsize=7.8, ha="right", va="center",
                color="#dbe4ee", style="italic")
    return (x, y, w, alto)

def centro(b):
    x, y, w, h = b
    return x + w / 2, y - h / 2

AZUL, GRIS, VERDE = "#1F3A5F", "#475569", "#0F766E"
fig, ax = plt.subplots(figsize=(16, 9.2))
ax.set_xlim(0, 16); ax.set_ylim(1.0, 10.2); ax.axis("off")

hecho = caja(ax, 5.9, 7.35, 4.2, "fact_venta_linea", [
    ("boleta_id, linea_id  (llave, degeneradas)", "pk"),
    ("fecha_key → dim_fecha", "fk"), ("cliente_key → dim_cliente", "fk"), ("tienda_key → dim_tienda", "fk"),
    ("producto_key → dim_producto", "fk"), ("campana_key → dim_campana", "fk"),
    ("cantidad, precio_unitario, descuento", "attr"), ("monto_linea, monto_linea_real_dic2025", "attr"),
    ("monto_esperado, linea_cuadra, es_devolucion", "attr")], AZUL, "POS")

dims = [
    caja(ax, 0.3, 9.9, 4.3, "dim_fecha", [("fecha_key (aaaammdd)", "pk"), ("fecha, mes, trimestre, día semana", "attr"),
         ("ipc_var_mensual, ipc_indice", "attr"), ("factor_a_pesos_dic2025", "attr")], GRIS, "API mindicador.cl"),
    caja(ax, 0.3, 7.0, 4.3, "dim_cliente", [("cliente_key (0 = Anónimo)", "pk"), ("segmento (tipo 1, nominal)", "attr"),
         ("tipo_cliente: Base heredada / Nuevo 2025", "attr"), ("fecha_alta, primera_visita_2025", "attr"),
         ("comuna, región, macrozona", "attr")], GRIS, "POS"),
    caja(ax, 0.3, 3.4, 4.3, "dim_tienda", [("tienda_key", "pk"), ("tienda, formato", "attr"), ("canal (Presencial / Web), es_online", "attr"),
         ("comuna, región, macrozona", "attr")], GRIS, "POS"),
    caja(ax, 11.4, 9.9, 4.3, "dim_producto", [("producto_key", "pk"), ("sku, nombre, categoría, subcategoría", "attr"),
         ("marca, proveedor, precio_lista, activo", "attr"), ("en_maestro (False = 'Sin maestro')", "attr")], GRIS, "Maestro CSV"),
    caja(ax, 11.4, 5.6, 4.3, "dim_campana", [("campana_key (0 = Sin campaña)", "pk"), ("nombre, tipo, descuento_pct", "attr"),
         ("fecha_ini, fecha_fin, canales", "attr")], GRIS, "Promociones JSON"),
]
cx, cy = centro(hecho)
for d in dims:
    dx, dy = centro(d)
    ax.annotate("", xy=(cx, cy), xytext=(dx, dy),
                arrowprops=dict(arrowstyle="-", color="#94a3b8", lw=1.2, shrinkA=60, shrinkB=95), zorder=0)

ax.text(8.0, hecho[1] - hecho[3] - 0.45,
        "Visita = boleta_id con monto neto > 0 (se agrega desde las líneas)", fontsize=9, color=VERDE,
        style="italic", ha="center")

ax.text(0.3, 1.2, "Grano de fact_venta_linea: una fila es un producto (SKU) vendido o devuelto en una boleta válida de 2025.",
        fontsize=10, fontweight="bold", color=AZUL)
fig.savefig(f"{SALIDAS}/modelo_dimensional.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 3.6 Qué quedó fuera del modelo y por qué
| Fuera del modelo | Por qué |
|---|---|
| Historial del segmento (tipo 2) | No existe en origen: la columna se sobrescribe (correo de Marketing). |
| Identidad de las compras anónimas y del hogar | No hay `cliente_id` ni identificador de hogar; se modela como un miembro "Anónimo". |
| Columnas `mes` y `region` de la boleta | Contradicen a la fecha y a la tienda (reglas B04 y B05). |
| Margen y costo como medidas | El costo solo existe en el maestro, con faltantes, y la rentabilidad es la P4, fuera de alcance. |
| Feriados y estacionalidad en `dim_fecha` | No se necesitan para la P1; se agregarían para analizar campañas (P3). |
| Tabla de inventario o stock | No hay fuente. |
| Dimensión de canal separada | El canal queda determinado por la tienda; se guarda como atributo de `dim_tienda` (decisión 7). |
| RUT y nombre del cliente | Minimización (Ley 21.719): la pregunta se responde con `cliente_id`. Quedan en `plata/clientes`. |
| Conteos de visitas en `dim_cliente` | Son medidas: se calculan desde el hecho, no se guardan en la dimensión. |
| Medio de pago | Se midió y no distingue comportamiento ni aporta a la P1 (decisión 8). Queda disponible en `plata/boletas`. |

In [ ]:
bitacora_df = pd.DataFrame(bitacora_supuestos)
bitacora_df.to_csv(f"{SALIDAS}/bitacora_supuestos.csv", index=False, encoding="utf-8-sig")
print(f"Bitácora completa (limpieza + modelo): {SALIDAS}/bitacora_supuestos.csv ({len(bitacora_df)} reglas)")

---
# Etapa 4 - Métrica: retención frente a reposición

**Pregunta (P1):** ¿La cadena está reteniendo a sus clientes o los está reponiendo con compradores de una sola vez?

## Ficha de la métrica

| Elemento | Definición |
|---|---|
| **Nombre** | Tasa de retención semestral de clientes activos |
| **Fórmula** | Clientes activos al 30-jun-2025 que siguen activos al 31-dic-2025 ÷ clientes activos al 30-jun-2025 |
| **Cliente activo al corte C** | Cliente identificado con al menos una **visita** en los últimos **W días** hasta C (inclusive) |
| **Visita** | Boleta con monto neto positivo (regla B07), agregada desde `fact_venta_linea` |
| **Filtros** | Solo clientes identificados (excluye boletas anónimas, regla B06); solo 2025 |
| **Unidad** | Porcentaje de clientes |
| **Ventana W** | **Base: 90 días.** Alternativa: 180 días. Referencia de la contraparte: 30 días |

**Indicadores complementarios** (mismos cortes y ventana):
- **Perdidos** = activos en junio que no están activos en diciembre.
- **Ganados** = activos en diciembre que no estaban activos en junio, separados en *nuevos* (clientes `Nuevo 2025` cuya
  primera visita es posterior al 30-jun) y *reactivados* (ya habían comprado antes, pero no estaban activos en junio).
- **% de la base de diciembre que es reposición** = ganados ÷ activos en diciembre.

### Por qué esta forma
Los dos cortes (30-jun y 31-dic) separan ventanas que **no se traslapan** con W = 90 (abr–jun frente a oct–dic) ni con
W = 180 (ene–jun frente a jul–dic). Una comparación mes a mes con ventana de 90 días contaría al mismo cliente como
activo tres meses seguidos e inflaría la retención. Con datos de un solo año, este diseño es el que permite calcular
ambas ventanas sin historia previa a 2025.

### Decisión de definición: la ventana
La contraparte no tiene una definición formal de cliente inactivo; su creencia es un mes ("Pero eso es una creencia",
[verbalizado]). Se elige **90 días** porque la mediana entre dos compras del mismo cliente es de 13 días y el 93% de las
pausas normales son menores a 90 días [medido, se recalcula abajo]: con 30 días, más de un cuarto de las pausas habituales
se leerían como abandono. **180 días** es la alternativa, más permisiva.

### Regla de lectura (fijada antes de calcular)
- **La cadena repone** si la mayoría (> 50%) de la base activa de diciembre son clientes ganados y no retenidos.
- **La cadena retiene** si la mayoría de la base de diciembre son clientes que ya estaban activos en junio.
- Si la conclusión cambia entre 90 y 180 días, la respuesta depende de la definición y así se informa.

In [ ]:
CORTE_1, CORTE_2 = pd.Timestamp("2025-06-30"), pd.Timestamp("2025-12-31")
W_BASE, W_ALT, W_REF = 90, 180, 30
FUENTE = "Fuente: POS Multitienda del Sur, 2025 (clientes identificados). Elaboración Grupo 3."

# Visitas de clientes identificados (una fila por boleta con monto neto > 0)
vis = visitas_por_boleta(fact_venta_linea)
vis = vis[vis.es_visita & ~vis.es_anonima].copy()
vis["fecha"] = pd.to_datetime(vis.fecha_key.astype(str), format="%Y%m%d")
vis = vis.merge(dim_cliente[["cliente_key", "tipo_cliente", "primera_visita_2025"]], on="cliente_key", how="left")
assert len(vis) == n_usadas, "Las visitas no cuadran con la cobertura"

# Evidencia para la ventana: días entre compras consecutivas del mismo cliente
gaps = vis.sort_values(["cliente_key", "fecha"]).groupby("cliente_key").fecha.diff().dt.days.dropna()
print(f"Días entre compras: mediana {gaps.median():.0f} | p75 {gaps.quantile(.75):.0f} | p90 {gaps.quantile(.9):.0f}")
for w in (W_REF, W_BASE, W_ALT):
    print(f"  Pausas normales mayores a {w} días: {(gaps > w).mean():.1%}")

In [ ]:
def activos(corte, w, datos=None):
    d = vis if datos is None else datos
    desde = corte - pd.Timedelta(days=w - 1)
    return set(d[(d.fecha >= desde) & (d.fecha <= corte)].cliente_key)

tipo_de = dim_cliente.set_index("cliente_key").tipo_cliente
primera_de = dim_cliente.set_index("cliente_key").primera_visita_2025

def retencion(w, datos=None):
    base = activos(CORTE_1, w, datos)
    final = activos(CORTE_2, w, datos)
    ret, perd, gan = base & final, base - final, final - base
    nuevos = {k for k in gan if tipo_de[k] == "Nuevo 2025" and primera_de[k] > CORTE_1}
    return {"ventana_dias": w, "activos_jun": len(base), "retenidos": len(ret), "perdidos": len(perd),
            "ganados": len(gan), "ganados_nuevos": len(nuevos), "ganados_reactivados": len(gan - nuevos),
            "activos_dic": len(final),
            "tasa_retencion": len(ret) / len(base) if base else np.nan,
            "pct_base_dic_reposicion": len(gan) / len(final) if final else np.nan,
            "_sets": {"base": base, "final": final, "ret": ret, "perd": perd, "gan": gan, "nuevos": nuevos}}

res = {w: retencion(w) for w in (W_REF, W_BASE, W_ALT)}
tabla_metrica = pd.DataFrame([{k: v for k, v in r.items() if k != "_sets"} for r in res.values()]).set_index("ventana_dias")
tabla_metrica["conclusion"] = np.where(tabla_metrica.pct_base_dic_reposicion > 0.5, "Repone", "Retiene")
tabla_metrica.to_csv(f"{SALIDAS}/metrica_retencion.csv")
display(tabla_metrica.style.format({"tasa_retencion": "{:.1%}", "pct_base_dic_reposicion": "{:.1%}"}))

In [ ]:
b, a = res[W_BASE], res[W_ALT]
print("RESULTADO")
print(f"  Ventana base (90 días):  retención {b['tasa_retencion']:.1%} | "
      f"{b['pct_base_dic_reposicion']:.1%} de la base de diciembre es reposición -> {tabla_metrica.loc[90, 'conclusion']}")
print(f"  Alternativa (180 días):  retención {a['tasa_retencion']:.1%} | "
      f"{a['pct_base_dic_reposicion']:.1%} de la base de diciembre es reposición -> {tabla_metrica.loc[180, 'conclusion']}")
print(f"  Diferencia por la definición: {100 * (a['tasa_retencion'] - b['tasa_retencion']):+.1f} puntos de retención, "
      f"{100 * (a['pct_base_dic_reposicion'] - b['pct_base_dic_reposicion']):+.1f} puntos de reposición")
print(f"  Referencia de la contraparte (30 días): retención {res[W_REF]['tasa_retencion']:.1%}")
misma = tabla_metrica.loc[90, "conclusion"] == tabla_metrica.loc[180, "conclusion"]
print("\n  La conclusión", "SE MANTIENE" if misma else "CAMBIA", "entre 90 y 180 días.")

## Segunda decisión de definición: qué cuenta como cliente nuevo
El acta original proponía tomar como fecha de adquisición la primera boleta, sin mirar la fecha de alta. Con datos de un
solo año, eso cuenta como "nuevo" a cualquier cliente heredado que vuelve después de una pausa. Se muestra cuántos
"nuevos" habría bajo cada criterio (regla M01).

In [ ]:
filas = []
for w in (W_BASE, W_ALT):
    s = res[w]["_sets"]
    nuevos_acta = {k for k in s["gan"] if primera_de[k] > CORTE_1}
    filas.append({"ventana_dias": w, "ganados": len(s["gan"]),
                  "nuevos (criterio del modelo: alta 2025)": len(s["nuevos"]),
                  "nuevos (criterio del acta: primera boleta)": len(nuevos_acta),
                  "diferencia": len(nuevos_acta) - len(s["nuevos"])})
display(pd.DataFrame(filas).set_index("ventana_dias"))

## Cortes de la métrica
- **Tipo de cliente:** base heredada frente a nuevos 2025.
- **Tienda principal:** la tienda donde el cliente hizo más visitas en la ventana de junio (pedido explícito de la
  contraparte: retención por local).
- **Mezcla de canales:** canales usados en la ventana de junio.
- **Segmento:** se muestra para confirmar que no discrimina (correo de Marketing).

Los cortes con menos de 30 clientes en la base de junio se marcan como poco confiables.

In [ ]:
def corte(atributo_por_cliente, w=W_BASE):
    s = res[w]["_sets"]
    df = pd.DataFrame({"cliente_key": sorted(s["base"])})
    df["grupo"] = df.cliente_key.map(atributo_por_cliente)
    df["retenido"] = df.cliente_key.isin(s["ret"])
    out = df.groupby("grupo").agg(activos_jun=("cliente_key", "size"), retenidos=("retenido", "sum"))
    out["tasa_retencion"] = out.retenidos / out.activos_jun
    out["poco_confiable"] = out.activos_jun < 30
    return out.sort_values("tasa_retencion")

# Ventana de junio (base) para asignar tienda principal y canales
desde_base = CORTE_1 - pd.Timedelta(days=W_BASE - 1)
v_base = vis[(vis.fecha >= desde_base) & (vis.fecha <= CORTE_1)].merge(
    dim_tienda[["tienda_key", "tienda", "canal"]], on="tienda_key")
tienda_principal = v_base.groupby("cliente_key").tienda.agg(lambda x: x.value_counts().index[0])
mezcla_canal = v_base.groupby("cliente_key").canal.agg(lambda x: " + ".join(sorted(x.unique())))
segmento_de = dim_cliente.set_index("cliente_key").segmento

cortes = {"tipo_cliente": corte(tipo_de), "tienda_principal": corte(tienda_principal),
          "mezcla_canal": corte(mezcla_canal), "segmento": corte(segmento_de)}
for nombre, t in cortes.items():
    t.to_csv(f"{SALIDAS}/retencion_por_{nombre}.csv")
    print(f"\nRetención a {W_BASE} días por {nombre}:")
    display(t.style.format({"tasa_retencion": "{:.1%}"}))

# Misma tabla por tienda con 180 días, para la sensibilidad
tienda_180 = corte(tienda_principal, W_ALT)

## Evidencia complementaria
**1. Recompra de los clientes nuevos.** De los clientes `Nuevo 2025`, qué porcentaje vuelve a comprar dentro de los 90
días siguientes a su primera visita, según el mes en que llegaron. Solo se incluyen meses cuyos clientes tuvieron los 90
días completos para volver (meses de llegada de enero a septiembre).

**2. Clientes con compra en cada mes**, separando base heredada y nuevos 2025, y la venta mensual en pesos de diciembre de
2025 (IPC de la API).

In [ ]:
nv = vis[vis.tipo_cliente == "Nuevo 2025"].copy()
nv["primera"] = nv.cliente_key.map(nv.groupby("cliente_key").fecha.min())
nv["dias"] = (nv.fecha - nv.primera).dt.days
segunda = nv[nv.dias > 0].groupby("cliente_key").dias.min()
coh = nv.groupby("cliente_key").primera.min().to_frame()
coh = coh[coh.primera.dt.month <= 9] # meses completos con 90 días de observación (llegada hasta el 30-sep)
coh["vuelve_90"] = coh.index.map(lambda k: segunda.get(k, np.inf) <= 90)
coh["mes_llegada"] = coh.primera.dt.month
recompra = coh.groupby("mes_llegada").agg(clientes_nuevos=("vuelve_90", "size"), vuelven_90=("vuelve_90", "sum"))
recompra["tasa_recompra_90"] = recompra.vuelven_90 / recompra.clientes_nuevos
recompra.to_csv(f"{SALIDAS}/recompra_90_nuevos_por_mes.csv")
display(recompra.style.format({"tasa_recompra_90": "{:.0%}"}))

vis["mes"] = vis.fecha.dt.month
mensual = (vis.groupby(["mes", "tipo_cliente"])
              .agg(clientes=("cliente_key", "nunique"), venta_real=("monto_boleta_real_dic2025", "sum"))
              .unstack("tipo_cliente"))
mensual.to_csv(f"{SALIDAS}/clientes_y_venta_mensual_por_tipo.csv")
display(mensual)

por_tienda_mes = (vis.merge(dim_tienda[["tienda_key", "tienda"]], on="tienda_key")
                     .groupby(["tienda", "mes"]).cliente_key.nunique().unstack("mes"))
por_tienda_mes.to_csv(f"{SALIDAS}/clientes_con_compra_por_tienda_y_mes.csv")

## Traducción a pesos
Montos en **pesos de diciembre de 2025** (`monto_linea_real_dic2025`, deflactado con el IPC de la API).

In [ ]:
s = res[W_BASE]["_sets"]
h1 = vis[vis.fecha <= CORTE_1]
gasto_perdidos_h1 = h1[h1.cliente_key.isin(s["perd"])].monto_boleta_real_dic2025.sum()
venta_h1 = h1.monto_boleta_real_dic2025.sum()
gasto_mensual_perdido = gasto_perdidos_h1 / len(s["perd"]) / 6
por_10pp = round(0.10 * len(s["base"]))
caso = {
    "clientes perdidos (90 días)": len(s["perd"]),
    "gasto 1er semestre de los perdidos ($ dic-2025)": round(gasto_perdidos_h1),
    "% de la venta identificada del 1er semestre": gasto_perdidos_h1 / venta_h1,
    "gasto mensual promedio por cliente perdido ($ dic-2025)": round(gasto_mensual_perdido),
    "clientes que equivalen a +10 pts de retención": por_10pp,
    "venta anual asociada a +10 pts de retención ($ dic-2025)": round(por_10pp * gasto_mensual_perdido * 12),
}
caso_df = pd.Series(caso, name="valor").to_frame()
caso_df.to_csv(f"{SALIDAS}/caso_de_negocio.csv")
display(caso_df.assign(valor=[f"{v:.1%}" if isinstance(v, float) and v < 1 else f"{v:,.0f}" for v in caso_df.valor]))

In [ ]:
registrar("K01", "métrica", "Ventana de actividad de 90 días (alternativa 180; referencia 30)",
          f"Mediana entre compras {gaps.median():.0f} días; {(gaps > 90).mean():.1%} de las pausas superan 90 días "
          f"y {(gaps > 30).mean():.1%} superan 30", len(res[W_BASE]['_sets']['base']),
          f"Con 180 días la retención sube a {res[W_ALT]['tasa_retencion']:.1%}; la conclusión "
          f"{'se mantiene' if misma else 'cambia'}", "verbalizado (creencia de 30 días) y medido")
registrar("K02", "métrica", "Cortes al 30-jun y al 31-dic para que las ventanas no se traslapen",
          "Comparaciones mes a mes con ventana de 90 días cuentan al mismo cliente varias veces", 2,
          "Diciembre es temporada alta: clientes que solo compran en fiestas pueden aparecer como activos", "inferido")
registrar("K03", "métrica", "Tienda principal = tienda con más visitas del cliente en la ventana de junio",
          "La mayoría de los clientes compra en más de un local; se asigna uno para cortar la retención",
          len(tienda_principal), "Un cliente repartido entre locales queda asignado solo a uno", "inferido")
pd.DataFrame(bitacora_supuestos).to_csv(f"{SALIDAS}/bitacora_supuestos.csv", index=False, encoding="utf-8-sig")

---
# Etapa 5 - Evidencia visual y caso de negocio

Se presentan cinco gráficos, cada uno con su unidad en el título y su fuente al pie, guardados en `salidas/graficos/` para el informe.
Todas las cifras salen de las tablas calculadas en la Etapa 4.

| # | Gráfico | Responde |
|---|---|---|
| G1 | Base activa al 30-jun y al 31-dic: retenidos, perdidos y ganados | La respuesta directa a la P1 |
| G2 | Sensibilidad a la ventana (30 / 90 / 180 días) | La decisión de definición (componente D) |
| G3 | Clientes con compra por mes: base heredada frente a nuevos | Cómo se vació la base y quién la reemplazó |
| G4 | Recompra a 90 días de los clientes nuevos, por mes de llegada | Si los clientes que reponen se quedan |
| G5 | Retención a 90 días por tienda principal | El corte por local que pidió la contraparte |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

GRAF = f"{SALIDAS}/graficos"
os.makedirs(GRAF, exist_ok=True)

# Colores (validada: azul, naranjo, aqua; el gris es neutro, no categórico)
AZUL, NARANJO, AQUA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#b8b6ae"
TXT, TXT2, GRILLA = "#0b0b0b", "#52514e", "#e7e6e2"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10.5, "axes.edgecolor": GRILLA, "axes.labelcolor": TXT2,
    "xtick.color": TXT2, "ytick.color": TXT2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlecolor": TXT, "axes.titlelocation": "left",
    "figure.facecolor": "white", "axes.facecolor": "white", "legend.frameon": False,
})

def pie(fig, nota=""):
    fig.text(0.01, -0.02, FUENTE + (" " + nota if nota else ""), fontsize=8.5, color=TXT2, ha="left", va="top")

def guardar(fig, nombre):
    fig.savefig(f"{GRAF}/{nombre}.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  -> {GRAF}/{nombre}.png")

## G1 - La base activa de diciembre es mayoritariamente reposición

In [ ]:
r = res[W_BASE]
fig, ax = plt.subplots(figsize=(10, 6))
x = [0, 1]
segmentos = [
    # (barra, valor, color, etiqueta)
    (0, r["retenidos"], AZUL, "Retenidos"), (0, r["perdidos"], GRIS, "Perdidos"),
    (1, r["retenidos"], AZUL, "Retenidos"), (1, r["ganados_nuevos"], NARANJO, "Ganados: nuevos"),
    (1, r["ganados_reactivados"], AQUA, "Ganados: reactivados"),
]
abajo = {0: 0, 1: 0}
for barra, val, col, etq in segmentos:
    ax.bar(barra, val, bottom=abajo[barra], width=0.5, color=col, edgecolor="white", linewidth=2)
    total = r["activos_jun"] if barra == 0 else r["activos_dic"]
    if val >= 60:
        ax.text(barra, abajo[barra] + val / 2, f"{etq}\n{val} ({val / total:.0%})", ha="center", va="center",
                fontsize=10, color="white" if col in (AZUL, NARANJO) else TXT, fontweight="bold")
    else:
        ax.text(barra + 0.28, abajo[barra] + val / 2, f"{etq}: {val} ({val / total:.0%})", ha="left", va="center",
                fontsize=9.5, color=TXT)
    abajo[barra] += val
for barra, total in [(0, r["activos_jun"]), (1, r["activos_dic"])]:
    ax.text(barra - 0.25, total + 12, f"{total} activos", ha="left", va="bottom", fontsize=11, fontweight="bold", color=TXT)
ax.set_xticks(x, ["Activos al 30 de junio", "Activos al 31 de diciembre"])
ax.set_ylabel("Clientes")
ax.set_ylim(0, r["activos_dic"] * 1.15)
ax.yaxis.grid(True, color=GRILLA); ax.set_axisbelow(True)
ax.set_title(f"Base de clientes activos, junio y diciembre de 2025 (n.º de clientes, ventana de {W_BASE} días)", pad=26)
ax.text(0, 1.02, f"Se retuvo el {r['tasa_retencion']:.0%} de la base de junio; el {r['pct_base_dic_reposicion']:.0%} "
        "de la base de diciembre son clientes que no estaban activos en junio", transform=ax.transAxes,
        fontsize=10, color=TXT2, va="bottom")
pie(fig, f"Activo = al menos una compra con monto positivo en los {W_BASE} días previos al corte.")
guardar(fig, "G1_base_activa_jun_dic")

## G2 - La conclusión no depende de la ventana elegida

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ventanas = [W_REF, W_BASE, W_ALT]
etiquetas = [f"{W_REF} días\n(creencia de la contraparte)", f"{W_BASE} días\n(definición base)",
             f"{W_ALT} días\n(alternativa)"]
xs = np.arange(len(ventanas)); ancho = 0.36
ret_v = [res[w]["tasa_retencion"] for w in ventanas]
rep_v = [res[w]["pct_base_dic_reposicion"] for w in ventanas]
b1 = ax.bar(xs - ancho / 2 - 0.01, ret_v, ancho, color=AZUL, label="Tasa de retención (base de junio que sigue activa)")
b2 = ax.bar(xs + ancho / 2 + 0.01, rep_v, ancho, color=NARANJO, label="Reposición (% de la base de diciembre que es ganada)")
for barras in (b1, b2):
    for bar in barras:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015, f"{bar.get_height():.0%}",
                ha="center", va="bottom", fontsize=10.5, color=TXT, fontweight="bold")
ax.axhline(0.5, color=TXT2, lw=1, ls="--")
ax.text(-0.55, 0.515, "Umbral de lectura: 50%", fontsize=9, color=TXT2, ha="left")
ax.set_xticks(xs, etiquetas)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.yaxis.grid(True, color=GRILLA); ax.set_axisbelow(True)
ax.legend(loc="upper left", fontsize=9.5, ncol=1, bbox_to_anchor=(0, 1.0))
ax.set_title("Retención y reposición según la ventana de actividad (% de clientes)")
pie(fig, "La reposición supera el 50% con las tres ventanas.")
guardar(fig, "G2_sensibilidad_ventana")

## G3 - La base heredada se vació durante el año y los nuevos la reemplazaron

In [ ]:
cli_mes = mensual["clientes"]
meses_txt = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
fig, ax = plt.subplots(figsize=(10, 5.6))
for col, color in [("Base heredada", AZUL), ("Nuevo 2025", NARANJO)]:
    serie = cli_mes[col].reindex(range(1, 13))
    ax.plot(serie.index, serie.values, color=color, lw=2.2, marker="o", markersize=6)
    nombre = "Base heredada (alta antes de 2025)" if col == "Base heredada" else "Nuevos 2025 (alta en 2025)"
    ax.text(12.15, serie.iloc[-1], f"{nombre}\n{serie.iloc[0]:.0f} en ene → {serie.iloc[-1]:.0f} en dic",
            color=TXT, fontsize=9.5, va="center")
ax.set_xticks(range(1, 13), meses_txt)
ax.set_xlim(0.7, 12.3)
ax.set_ylabel("Clientes con al menos una compra en el mes")
ax.yaxis.grid(True, color=GRILLA); ax.set_axisbelow(True)
ax.set_title("Clientes con compra en cada mes de 2025, por tipo de cliente (n.º de clientes)")
pie(fig)
guardar(fig, "G3_clientes_por_mes_tipo")

## G4 - Los clientes que llegan después vuelven menos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.4))
rc = recompra.reindex(range(1, 10))
barras = ax.bar(rc.index, rc.tasa_recompra_90, width=0.6, color=NARANJO)
for bar, (_, fila) in zip(barras, rc.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012, f"{fila.tasa_recompra_90:.0%}",
            ha="center", va="bottom", fontsize=10.5, fontweight="bold", color=TXT)
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={fila.clientes_nuevos:.0f}", ha="center", va="bottom",
            fontsize=8.5, color="white")
ax.set_xticks(range(1, 10), meses_txt[:9])
ax.set_xlabel("Mes de la primera compra")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.yaxis.grid(True, color=GRILLA); ax.set_axisbelow(True)
ax.set_title("Clientes nuevos que vuelven a comprar dentro de 90 días, por mes de llegada (% de clientes)")
pie(fig, "Solo clientes con alta en 2025 y meses con 90 días completos de observación.")
guardar(fig, "G4_recompra_90_nuevos")

## G5 - La retención varía entre locales

In [ ]:
t90 = cortes["tienda_principal"].sort_values("tasa_retencion")
t180 = tienda_180.reindex(t90.index)
fig, ax = plt.subplots(figsize=(10, 6))
ys = np.arange(len(t90))
ax.barh(ys, t90.tasa_retencion, height=0.55, color=AZUL, label=f"{W_BASE} días (base)")
ax.scatter(t180.tasa_retencion, ys, s=60, facecolor="white", edgecolor=TXT2, linewidth=1.6, zorder=3,
           label=f"{W_ALT} días (alternativa)")
for y, (tienda, fila) in zip(ys, t90.iterrows()):
    ax.text(fila.tasa_retencion - 0.01, y, f"{fila.tasa_retencion:.0%}", va="center", ha="right", fontsize=9.5,
            color="white", fontweight="bold")
ax.axvline(res[W_BASE]["tasa_retencion"], color=TXT2, lw=1, ls="--")
ax.text(res[W_BASE]["tasa_retencion"] + 0.005, len(t90) - 0.4, f"Cadena: {res[W_BASE]['tasa_retencion']:.0%}",
        fontsize=9, color=TXT2)
ax.set_yticks(ys, [f"{t} (n={n})" for t, n in zip(t90.index, t90.activos_jun)])
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlim(0, 1)
ax.xaxis.grid(True, color=GRILLA); ax.set_axisbelow(True)
ax.legend(loc="lower right", fontsize=9.5)
ax.set_title(f"Tasa de retención semestral por tienda principal del cliente (% de clientes)")
pie(fig, "n = clientes activos al 30-jun asignados a la tienda; con n entre 30 y 50, diferencias menores a ~15 pts no son concluyentes.")
guardar(fig, "G5_retencion_por_tienda")

## Resumen
Las cifras se imprimen desde las variables del notebook, para usar exactamente los números que reproduce
el código.

In [ ]:
b, a = res[W_BASE], res[W_ALT]
c = caso
mejor = cortes["tienda_principal"].tasa_retencion.idxmax(); peor = cortes["tienda_principal"].tasa_retencion.idxmin()
her, nue = cortes["tipo_cliente"].loc["Base heredada"], cortes["tipo_cliente"].loc["Nuevo 2025"]
print(f'''RESPUESTA A LA P1
La cadena está REPONIENDO clientes más que reteniéndolos.
- Con ventana de {W_BASE} días, de {b["activos_jun"]} clientes activos al 30 de junio siguen activos {b["retenidos"]} al 31 de diciembre:
  retención de {b["tasa_retencion"]:.1%}. Se perdieron {b["perdidos"]}.
- De los {b["activos_dic"]} activos en diciembre, {b["ganados"]} ({b["pct_base_dic_reposicion"]:.1%}) no estaban activos en junio
  ({b["ganados_nuevos"]} nuevos y {b["ganados_reactivados"]} reactivados).
- Con {W_ALT} días: retención {a["tasa_retencion"]:.1%} y reposición {a["pct_base_dic_reposicion"]:.1%}. La conclusión se mantiene;
  la definición mueve la retención en {100 * (a["tasa_retencion"] - b["tasa_retencion"]):+.1f} puntos.
- Base heredada: retención {her.tasa_retencion:.1%} frente a {nue.tasa_retencion:.1%} de los nuevos 2025.
- Recompra a 90 días de los nuevos: {recompra.tasa_recompra_90.iloc[0]:.0%} en enero frente a {recompra.tasa_recompra_90.iloc[-1]:.0%} en septiembre.
- Por local: de {cortes["tienda_principal"].tasa_retencion.min():.0%} ({peor}) a {cortes["tienda_principal"].tasa_retencion.max():.0%} ({mejor}).

EN PESOS (diciembre de 2025)
- Los {c["clientes perdidos (90 días)"]} clientes perdidos gastaron ${c["gasto 1er semestre de los perdidos ($ dic-2025)"]:,.0f} en el primer semestre,
  el {c["% de la venta identificada del 1er semestre"]:.0%} de la venta identificada de ese período (${c["gasto mensual promedio por cliente perdido ($ dic-2025)"]:,.0f} al mes por cliente).
- Cada 10 puntos de retención equivalen a ~{c["clientes que equivalen a +10 pts de retención"]} clientes y ~${c["venta anual asociada a +10 pts de retención ($ dic-2025)"]:,.0f} de venta anual.
''')

In [ ]:
pd.DataFrame(bitacora_supuestos).to_csv(f"{SALIDAS}/bitacora_supuestos.csv", index=False, encoding="utf-8-sig")
print("Archivos generados en salidas/:")
for raiz, _, archivos in os.walk(SALIDAS):
    for a_ in sorted(archivos):
        print("  ", os.path.join(raiz, a_))